# Homework 3 · Clasificación (ML Zoomcamp 2026) Jaime Juárez


## Importación de Librerías y Carga de datos:

In [1]:
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score, accuracy_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

# URL indicada en homework.md: se lee directo a un DataFrame, sin guardar CSV local.
url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"
df = pd.read_csv(url)

print(f"Dataset cargado: {df.shape[0]:,} filas y {df.shape[1]} columnas")
display(df.head())

Dataset cargado: 5,000 filas y 9 columnas


,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


## Revisar y tratar los valores faltantes

In [2]:
print("Valores faltantes por columna (antes de limpiar):")
display(df.isna().sum().to_frame("faltantes"))

# Separamos columnas predictoras de la variable objetivo.
objetivo = "converted"
X = df.drop(columns=[objetivo]).copy()
y = df[objetivo].copy()

categoricas = X.select_dtypes(include=["object", "category"]).columns.tolist()
numericas = X.select_dtypes(include=["number"]).columns.tolist()

# Instrucciones: categóricas -> 'NA'; numéricas -> 0.0.
X[categoricas] = X[categoricas].fillna("NA")
X[numericas] = X[numericas].fillna(0.0)

# Reconstruimos un DataFrame limpio con la columna objetivo para dividirlo.
datos = X.assign(converted=y)
assert not X.isna().any().any(), "Quedaron nulos sin imputar"
print("Nulos en las variables predictoras después de limpiar:", X.isna().sum().sum())

Valores faltantes por columna (antes de limpiar):


,faltantes
lead_source,148
industry,240
employment_status,194
location,208
annual_income,369
number_of_courses_viewed,0
interaction_count,0
lead_score,35
converted,0


Nulos en las variables predictoras después de limpiar: 0


## 3. Preguntas 1 y 2 · Moda y correlación

In [3]:
# P1: valor más frecuente en industry.
respuesta_1 = X["industry"].mode().iloc[0]
print("P1. Moda de industry:", respuesta_1)

# P2: matriz de correlaciones SOLO entre variables numéricas predictoras.
correlaciones = X[numericas].corr()
print("\nP2. Matriz de correlación (numéricas):")
display(correlaciones.round(3))

# El enunciado limita la comparación a estos 4 pares.
pares = [
    ("interaction_count", "lead_score"),
    ("number_of_courses_viewed", "lead_score"),
    ("number_of_courses_viewed", "interaction_count"),
    ("annual_income", "interaction_count"),
]
valores = {(a, b): abs(correlaciones.loc[a, b]) for a, b in pares}
respuesta_2 = max(valores, key=valores.get)
print("P2. Mayor correlación:", " y ".join(respuesta_2),
      f"(r = {correlaciones.loc[respuesta_2]:.3f})")

P1. Moda de industry: technology

P2. Matriz de correlación (numéricas):


,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000,0.161,0.123,0.229
number_of_courses_viewed,0.161,1.000,0.722,0.757
interaction_count,0.123,0.722,1.000,0.916
lead_score,0.229,0.757,0.916,1.000


P2. Mayor correlación: interaction_count y lead_score (r = 0.916)


## 4. Dividir los datos y responder la pregunta 3

In [4]:
# Divisiones EXACTAS indicadas en homework.md (60 % / 20 % / 20 %).
df_full_train, df_test = train_test_split(datos, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

# Quitar converted del conjunto de variables independientes.
y_train, y_val = df_train["converted"], df_val["converted"]
X_train = df_train.drop(columns=["converted"])
X_val = df_val.drop(columns=["converted"])
print(f"Entrenamiento: {len(X_train)} | Validación: {len(X_val)} | Test: {len(df_test)}")

# P3: información mutua usando EXCLUSIVAMENTE el conjunto de entrenamiento.
mi = {col: mutual_info_score(y_train, X_train[col]) for col in categoricas}
mi_tabla = pd.Series(mi, name="mutual_information").sort_values(ascending=False)
display(mi_tabla.round(2).to_frame())
respuesta_3 = mi_tabla.idxmax()
print("P3. Variable categórica con mayor información mutua:", respuesta_3)

Entrenamiento: 3000 | Validación: 1000 | Test: 1000


,mutual_information
lead_source,0.03
employment_status,0.02
industry,0.00
location,0.00


P3. Variable categórica con mayor información mutua: lead_source


## 5. Pregunta 4 · Regresión logística con one-hot encoding

In [5]:
# Función reutilizable para Q4, Q5 y Q6.
def entrenar_y_evaluar(columnas, C=1.0):
    # DictVectorizer aplica one-hot encoding a las columnas de texto.
    codificador = DictVectorizer(sparse=False)
    X_ent = codificador.fit_transform(X_train[columnas].to_dict(orient="records"))
    X_valid = codificador.transform(X_val[columnas].to_dict(orient="records"))

    modelo = LogisticRegression(
        solver="liblinear", C=C, max_iter=1000, random_state=42
    )
    modelo.fit(X_ent, y_train)
    pred = modelo.predict(X_valid)
    return accuracy_score(y_val, pred)

columnas = X_train.columns.tolist()  # Todas las variables predictoras.
accuracy_base = entrenar_y_evaluar(columnas, C=1.0)
respuesta_4 = round(accuracy_base, 2)
print(f"P4. Accuracy sin redondear: {accuracy_base:.6f}")
print(f"P4. Respuesta (2 decimales): {respuesta_4:.2f}")

P4. Accuracy sin redondear: 0.645000
P4. Respuesta (2 decimales): 0.65


## 6. Pregunta 5 · Eliminación de variables

In [6]:
# Comparamos el accuracy original contra el accuracy sin cada columna indicada.
excluir = ["lead_source", "number_of_courses_viewed", "interaction_count"]
resultados_5 = []
for columna in excluir:
    accuracy_sin = entrenar_y_evaluar(
        [c for c in columnas if c != columna], C=1.0
    )
    diferencia = accuracy_base - accuracy_sin  # Con signo, como indica el enunciado.
    resultados_5.append({
        "variable_excluida": columna,
        "accuracy_sin_variable": accuracy_sin,
        "diferencia_original_menos_sin": diferencia,
    })

tabla_5 = pd.DataFrame(resultados_5)
display(tabla_5.round(5))

# El texto dice 'menor diferencia' y aclara que puede ser negativa.
respuesta_5 = tabla_5.loc[
    tabla_5["diferencia_original_menos_sin"].idxmin(), "variable_excluida"
]
print("P5. Menor diferencia con signo:", respuesta_5)

# Nota de auditoría: el verificador de referencia del repositorio usa valor
# absoluto. Se muestra también para detectar si ambos criterios difieren.
respuesta_5_abs = tabla_5.loc[
    tabla_5["diferencia_original_menos_sin"].abs().idxmin(), "variable_excluida"
]
if respuesta_5 != respuesta_5_abs:
    print("AVISO: menor cambio en valor absoluto:", respuesta_5_abs,
          "(criterio distinto al texto del MD)")

,variable_excluida,accuracy_sin_variable,diferencia_original_menos_sin
0,lead_source,0.642,0.003
1,number_of_courses_viewed,0.643,0.002
2,interaction_count,0.601,0.044


P5. Menor diferencia con signo: number_of_courses_viewed


## 7. Pregunta 6 · Regularización (parámetro C)

In [7]:
# Mismo modelo de Q4, cambiando C. En empates se escoge el C menor.
C_valores = [0.000001, 0.00001, 0.0001, 0.001]
resultados_6 = pd.DataFrame([
    {"C": C, "accuracy": entrenar_y_evaluar(columnas, C=C)}
    for C in C_valores
])
resultados_6["accuracy_redondeado"] = resultados_6["accuracy"].round(3)
display(resultados_6)

# Priorizamos accuracy a 3 decimales y, en empate, el C más pequeño.
mejor_valor = resultados_6["accuracy_redondeado"].max()
respuesta_6 = resultados_6.loc[
    resultados_6["accuracy_redondeado"] == mejor_valor, "C"
].min()
print(f"P6. Mejor C: {respuesta_6:g} | accuracy = {mejor_valor:.3f}")

,C,accuracy,accuracy_redondeado
0,0.000001,0.598,0.598
1,0.000010,0.598,0.598
2,0.000100,0.613,0.613
3,0.001000,0.645,0.645


P6. Mejor C: 0.001 | accuracy = 0.645
